In [1]:
import os
import gc
import torch
import numpy as np
import pandas as pd
import shutil
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
from torch.utils.data import Dataset

model_nickname = "mBERT"
model_core_name = "bert-base-multilingual-cased"

kb_paths = {
    "kb1": "/kaggle/input/datasets/kghangco/vn-fb-news-dataset/01_Raw_Data/01_Raw_Data",
    "kb2": "/kaggle/input/datasets/kghangco/vn-fb-news-dataset/02_Basic_Clean/02_Basic_Clean",
    "kb3": "/kaggle/input/datasets/kghangco/vn-fb-news-dataset/03_Full_Clean/03_Full_Clean",
    "kb4": "/kaggle/input/datasets/kghangco/vn-fb-news-dataset/04_No_Stopwords/04_No_Stopwords",
    "kb5": "/kaggle/input/datasets/kghangco/vn-fb-news-dataset/05_Balanced/05_Balanced"
}

all_unique_labels = sorted(["T01", "T02", "T03", "T04", "T05", "T06", "T07", "T08", "T09", "T10", "T11", "T12", "T13", "T14", "T15", "T16", "T17"])
label2id = {label: i for i, label in enumerate(all_unique_labels)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(all_unique_labels)

mbert_results = []

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"Accuracy": acc, "Precision": precision, "Recall": recall, "F1-macro": f1}

print(f" Khởi động Tokenizer cho: {model_nickname}...")
tokenizer = AutoTokenizer.from_pretrained(model_core_name)

for kb_name, folder_path in kb_paths.items():
    print("\n" + "="*80)
    print(f" HUẤN LUYỆN [{model_nickname}] - KỊCH BẢN: [{kb_name.upper()}]")
    print("="*80)
    
    train_path = os.path.join(folder_path, f"{kb_name}_train.csv")
    val_path = os.path.join(folder_path, f"{kb_name}_val.csv")
    test_path = os.path.join(folder_path, f"{kb_name}_test.csv")
    
    if not os.path.exists(train_path):
        print(f" Không tìm thấy file hệ thống cho {kb_name.upper()}. Bỏ qua.")
        continue
        
    df_train = pd.read_csv(train_path)
    df_val = pd.read_csv(val_path)
    df_test = pd.read_csv(test_path)
    
    available_cols = df_train.columns.tolist()
    if kb_name == 'kb4' and 'text_nosw' in available_cols:
        text_col = 'text_nosw'
    elif kb_name in ['kb2', 'kb3', 'kb5'] and 'text_clean' in available_cols:
        text_col = 'text_clean'
    elif 'post_content_for_labeling' in available_cols:
        text_col = 'post_content_for_labeling'
    else:
        text_col = available_cols[19] 
        
    print(f" Quyết định: Cột văn bản được chọn là [{text_col}]")
    
    y_train = df_train['topic_label_id'].map(label2id).astype(int).tolist()
    y_val = df_val['topic_label_id'].map(label2id).astype(int).tolist()
    y_test = df_test['topic_label_id'].map(label2id).astype(int).tolist()
    
    train_encodings = tokenizer(df_train[text_col].astype(str).tolist(), truncation=True, padding=True, max_length=256)
    val_encodings = tokenizer(df_val[text_col].astype(str).tolist(), truncation=True, padding=True, max_length=256)
    test_encodings = tokenizer(df_test[text_col].astype(str).tolist(), truncation=True, padding=True, max_length=256)
    
    train_dataset = TextDataset(train_encodings, y_train)
    val_dataset = TextDataset(val_encodings, y_val)
    test_dataset = TextDataset(test_encodings, y_test)
    
    model = AutoModelForSequenceClassification.from_pretrained(model_core_name, num_labels=num_labels)
    
    tmp_output_dir = f"/kaggle/working/tmp_mbert_{kb_name}"
    
    training_args = TrainingArguments(
        output_dir=tmp_output_dir,
        num_train_epochs=10,                
        per_device_train_batch_size=16,   
        per_device_eval_batch_size=16,
        learning_rate=2e-5,               
        weight_decay=0.01,
        lr_scheduler_type="linear",       
        warmup_ratio=0.1,                                           
        eval_strategy="epoch",           
        save_strategy="epoch",            
        save_total_limit=1,                
        load_best_model_at_end=True,   
        metric_for_best_model="F1-macro",
        greater_is_better=True,
        report_to="none"
    )
    
    trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=val_dataset, compute_metrics=compute_metrics)
    trainer.train()
    
    print(f" Đang đóng gói và xuất file Weight Best Scenario cho mBERT_{kb_name.upper()}...")
    final_model_save_path = f"/kaggle/working/mbert_best_model_{kb_name}"
    
    trainer.save_model(final_model_save_path)
    tokenizer.save_pretrained(final_model_save_path)
    
    shutil.make_archive(f"/kaggle/working/mbert_best_model_{kb_name}", 'zip', final_model_save_path)
    
    if os.path.exists(tmp_output_dir):
        shutil.rmtree(tmp_output_dir)
    if os.path.exists(final_model_save_path):
        shutil.rmtree(final_model_save_path)
    print(f"Đã đóng gói thành công file: mbert_best_model_{kb_name}.zip")
    
    print(f"\nĐang đánh giá trên tập Test của kịch bản {kb_name.upper()}...")
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=1)
    
    target_names = [str(id2label[i]) for i in range(num_labels)]
    print(classification_report(y_test, y_pred, target_names=target_names, digits=4))
    
    test_metrics = compute_metrics((predictions.predictions, y_test))
    
    mbert_results.append({
        "Mô hình": model_nickname, 
        "Kịch bản dữ liệu": "KB5 (Oversampling)" if kb_name == "kb5" else kb_name.upper(),
        "Accuracy": round(test_metrics["Accuracy"], 4), 
        "Precision": round(test_metrics["Precision"], 4),
        "Recall": round(test_metrics["Recall"], 4), 
        "F1-macro": round(test_metrics["F1-macro"], 4)
    })
    
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

df_mbert = pd.DataFrame(mbert_results)
df_mbert.to_csv("/kaggle/working/mbert_report.csv", index=False)
print("\n" + "="*65)
print(" HOÀN THÀNH MBERT TOÀN DIỆN! Đã lưu file báo cáo và 5 file weights zip.")
print("="*65)
print(df_mbert.to_string(index=False))

 Khởi động Tokenizer cho: mBERT...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

 Đang đóng gói và xuất file Weight Best Scenario cho mBERT_KB5...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã đóng gói thành công file: mbert_best_model_kb5.zip

Đang đánh giá trên tập Test của kịch bản KB5...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

         T01     0.8462    0.7586    0.8000        58
         T02     0.6310    0.8030    0.7067        66
         T03     0.8025    0.8182    0.8103       154
         T04     0.6757    0.7576    0.7143        33
         T05     0.6296    0.9444    0.7556        18
         T06     0.6250    0.5882    0.6061        17
         T07     0.4286    0.4286    0.4286         7
         T08     0.8571    0.8571    0.8571        14
         T09     0.6122    0.7895    0.6897        38
         T10     0.9058    0.7102    0.7962       176
         T11     1.0000    0.8148    0.8980        27
         T12     0.3214    0.4286    0.3673        21
         T13     0.6633    0.6019    0.6311       108
         T14     0.5000    0.5556    0.5263         9
         T15     0.4118    0.4118    0.4118        17
         T16     0.8810    0.8222    0.8506        45
         T17     0.6000    0.8000    0.6857        30

    accuracy              